# Campus Point Wave Model Bias Correction

**Aadhav Ravi · UCSB · 2025–2026**

CDIP's MOP B0385 nearshore wave model systematically underpredicts significant wave height (Hs) at Campus Point by approximately 20%. This notebook builds and validates a direction-stratified bias correction model that reduces that error to near zero on held-out data.

The correction is grounded in O'Reilly et al. (2016), which showed that Santa Barbara Channel model errors are driven by wave diffraction through Channel Islands gaps — a direction-dependent physical mechanism. We test three correction approaches against a winter test set and export the best-performing model for use in a real-time forecast pipeline.

**Data sources:**
- **SPOT-1644** — Sofar buoy deployed by Nick Nidzieko at Campus Point (10m depth). Ground truth observations, ~30 min cadence.
- **MOP B0385** — CDIP's operational MOP nearshore wave model at Campus Point. Hourly hindcast + nowcast via THREDDS OPeNDAP.
- **Harvest (CDIP 071)** — Offshore reference buoy at 548m depth, west of Channel Islands. Used to classify offshore swell direction.
- **WaveWatch III** — NOAA global wave model (PacIOOS ERDDAP). Used to assess WW3 direction as a drop-in replacement for Harvest in the real-time pipeline.

**Reference:** O'Reilly et al. (2016), *The California coastal wave monitoring and prediction system*, Coastal Engineering 116, 118–132.

---
## 0. Imports

In [ ]:
import glob
import json
import urllib.request
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import xarray as xr
from scipy import stats
from scipy.interpolate import interp1d
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

pd.set_option('display.float_format', '{:.3f}'.format)
plt.rcParams['figure.dpi'] = 120

---
## 1. Data Ingestion

### 1a. SPOT-1644 — ground truth observations

Monthly CSVs are concatenated and deduplicated. The buoy was deployed on March 21, 2025, so the February file is nearly empty and skipped. Sub-hourly observations (~30 min cadence) are resampled to hourly means in Section 1d.

In [ ]:
def load_sofar(filepath):
    """Load a Sofar SPOT CSV and return a clean DataFrame indexed by UTC time."""
    df = pd.read_csv(filepath)
    df['time'] = pd.to_datetime(df['Epoch Time'], unit='s', utc=True)
    df = df[df['Significant Wave Height (m)'] != '-'].copy()
    df['Hs']  = pd.to_numeric(df['Significant Wave Height (m)'], errors='coerce')
    df['Tp']  = pd.to_numeric(df['Peak Period (s)'],             errors='coerce')
    df['Dp']  = pd.to_numeric(df['Peak Direction (deg)'],        errors='coerce')
    df['Dir'] = pd.to_numeric(df['Mean Direction (deg)'],        errors='coerce')
    df = df.dropna(subset=['Hs', 'Tp', 'Dp'])
    return df.set_index('time').sort_index()[['Hs', 'Tp', 'Dp', 'Dir']]

CSV_DIR = 'DATA/'
files = sorted(glob.glob(f'{CSV_DIR}SPOT-1644_*.csv'))

frames = []
for f in files:
    df = load_sofar(f)
    if len(df) == 0:
        print(f'SKIP (empty): {f}')
        continue
    frames.append(df)
    print(f'{f.split("/")[-1]}: {len(df)} obs  [{df.index[0].date()} -> {df.index[-1].date()}]')

spot = pd.concat(frames)
spot = spot[~spot.index.duplicated(keep='first')].sort_index()

print(f'\nTotal: {len(spot):,} observations')
print(f'Range: {spot.index[0].date()} -> {spot.index[-1].date()}')

### 1b. MOP B0385 — CDIP nearshore wave model

CDIP's MOP model for B0385 is split across two THREDDS endpoints: a hindcast (historical reanalysis) and a nowcast (near-real-time). Both are loaded and concatenated, with hindcast taking priority where timestamps overlap.

In [ ]:
HINDCAST_URL = "https://thredds.cdip.ucsd.edu/thredds/dodsC/cdip/model/MOP_alongshore/B0385_hindcast.nc"
NOWCAST_URL  = "https://thredds.cdip.ucsd.edu/thredds/dodsC/cdip/model/MOP_alongshore/B0385_nowcast.nc"

t_start = np.datetime64('2025-03-21')
t_end   = np.datetime64('2026-04-06')

def load_mop(url, t_start, t_end):
    """Load MOP bulk wave parameters from a CDIP THREDDS endpoint."""
    ds    = xr.open_dataset(url, engine='pydap')
    times = ds['waveTime'].values
    mask  = (times >= t_start) & (times <= t_end)
    df = pd.DataFrame({
        'Hs': ds['waveHs'].values[mask],
        'Dp': ds['waveDp'].values[mask],
        'Tp': ds['waveTp'].values[mask],
    }, index=pd.DatetimeIndex(times[mask], tz='UTC'))
    return df[(df['Hs'] > 0) & (df['Hs'] < 20)]

mop_hind = load_mop(HINDCAST_URL, t_start, t_end)
mop_now  = load_mop(NOWCAST_URL,  t_start, t_end)

print(f'Hindcast: {len(mop_hind):,} records  [{mop_hind.index[0].date()} -> {mop_hind.index[-1].date()}]')
print(f'Nowcast:  {len(mop_now):,} records   [{mop_now.index[0].date()} -> {mop_now.index[-1].date()}]')

mop = pd.concat([mop_hind, mop_now])
mop = mop[~mop.index.duplicated(keep='first')].sort_index()

print(f'\nMOP combined: {len(mop):,} records  [{mop.index[0].date()} -> {mop.index[-1].date()}]')

### 1c. Harvest buoy (CDIP 071) — offshore swell direction

Harvest sits at 548m depth well offshore and west of the Channel Islands, measuring the true incident swell direction before any island shadowing occurs. Only QC-flagged good data (waveFlagPrimary == 1) is retained.

In [ ]:
HARVEST_URL = "https://thredds.cdip.ucsd.edu/thredds/dodsC/cdip/realtime/071p1_rt.nc"

ds_h    = xr.open_dataset(HARVEST_URL, engine='pydap')
times_h = ds_h['waveTime'].values
flag_h  = ds_h['waveFlagPrimary'].values

mask_h = (times_h >= t_start) & (times_h <= t_end) & (flag_h == 1)

harvest = pd.DataFrame({
    'Hs': ds_h['waveHs'].values[mask_h],
    'Dp': ds_h['waveDp'].values[mask_h],
    'Tp': ds_h['waveTp'].values[mask_h],
}, index=pd.DatetimeIndex(times_h[mask_h], tz='UTC'))

harvest = harvest[(harvest['Hs'] > 0) & (harvest['Hs'] < 20)]

print(f'Harvest: {len(harvest):,} records')
print(f'Range:   {harvest.index[0].date()} -> {harvest.index[-1].date()}')
print(f'Hs:      {harvest["Hs"].min():.2f}m - {harvest["Hs"].max():.2f}m')
print(f'Dp:      {harvest["Dp"].min():.0f} - {harvest["Dp"].max():.0f} deg')

### 1d. Three-way alignment

SPOT observations are resampled to hourly means, then joined with MOP and Harvest on the hour using an inner join. A timestamp only survives if all three sources have valid data — no imputation.

Observations with SPOT Hs > 2m are excluded. These correspond to rare extreme south swell events where MOP has essentially zero skill due to severe Channel Islands diffraction. They represent only 0.3% of the dataset but disproportionately degrade bulk R² metrics.

In [ ]:
spot_hr = spot.resample('1h').mean()

aligned = (
    spot_hr
    .rename(columns=lambda c: c + '_spot')
    .join(mop.rename(columns=lambda c: c + '_mop'),         how='inner')
    .join(harvest.rename(columns=lambda c: c + '_harvest'), how='inner')
)

aligned = aligned.dropna()
aligned = aligned[aligned['Hs_spot'] < 2.0]

print(f'Aligned pairs: {len(aligned):,}')
print(f'Range: {aligned.index[0].date()} -> {aligned.index[-1].date()}')
print()
print(aligned.head(3))

---
## 2. Baseline Validation

We characterize MOP's raw skill against SPOT observations using the metrics from O'Reilly et al. (2016) Eq. 7: R2, bias, and bias-removed RMSE. The dataset is then stratified by offshore swell direction at Harvest to identify which swell geometries drive the largest errors.

In [ ]:
def skill_metrics(obs, pred, label=''):
    """Compute R2, bias, and bias-removed RMSE following O'Reilly et al. (2016) Eq. 7."""
    n        = len(obs)
    bias     = np.mean(pred - obs)
    rmse     = np.sqrt(np.mean((pred - obs - bias)**2))
    ss       = 1 - np.sum((pred - obs)**2) / np.sum((obs - np.mean(obs))**2)
    pct_bias = 100 * bias / np.mean(obs)
    pct_rmse = 100 * rmse / np.mean(obs)
    print(f'\n--- {label} ---')
    print(f'N        = {n:,}')
    print(f'Mean obs = {np.mean(obs):.3f}m   Mean pred = {np.mean(pred):.3f}m')
    print(f'R2       = {ss:.3f}')
    print(f'Bias     = {bias:.3f}m  ({pct_bias:.1f}%)')
    print(f'RMSE     = {rmse:.3f}m  ({pct_rmse:.1f}%)')
    return {'label': label, 'n': n, 'R2': ss,
            'bias': bias, 'pct_bias': pct_bias,
            'rmse': rmse, 'pct_rmse': pct_rmse}

obs  = aligned['Hs_spot'].values
pred = aligned['Hs_mop'].values

overall = skill_metrics(obs, pred, 'Full dataset -- MOP B0385 vs SPOT-1644')

month  = aligned.index.month
winter = (month >= 10) | (month <= 3)
summer = ~winter

skill_metrics(obs[winter], pred[winter], 'Winter (Oct-Mar)')
skill_metrics(obs[summer], pred[summer], 'Summer (Apr-Sep)')

In [ ]:
# Direction-stratified bias table.
# Five bins span the swell window for the Santa Barbara Channel (150-330 deg).

offshore_bins = {
    'NW  (290-330)': (aligned['Dp_harvest'] >= 290) & (aligned['Dp_harvest'] < 330),
    'WNW (260-290)': (aligned['Dp_harvest'] >= 260) & (aligned['Dp_harvest'] < 290),
    'W   (230-260)': (aligned['Dp_harvest'] >= 230) & (aligned['Dp_harvest'] < 260),
    'SW  (200-230)': (aligned['Dp_harvest'] >= 200) & (aligned['Dp_harvest'] < 230),
    'S   (150-200)': (aligned['Dp_harvest'] >= 150) & (aligned['Dp_harvest'] < 200),
}

print(f'{"Direction bin":<20} {"N":>6} {"Hs_obs":>8} {"Hs_mop":>8} {"Bias(m)":>9} {"Bias(%)": >9} {"R2":>8}')
print('-' * 72)

for name, mask in offshore_bins.items():
    sub = aligned[mask]
    if len(sub) < 10:
        continue
    o = sub['Hs_spot'].values
    p = sub['Hs_mop'].values
    bias = np.mean(p - o)
    ss   = 1 - np.sum((p - o)**2) / np.sum((o - np.mean(o))**2)
    print(f'{name:<20} {len(sub):>6} {np.mean(o):>8.3f} {np.mean(p):>8.3f} '
          f'{bias:>9.3f} {100*bias/np.mean(o):>9.1f} {ss:>8.3f}')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 7))
fig.suptitle('MOP B0385 vs SPOT-1644 -- Campus Point (Mar 2025 - Apr 2026)',
             fontweight='bold')

ax = axes[0]
ax.plot(aligned.index, aligned['Hs_spot'], lw=0.7, alpha=0.9,
        color='steelblue', label='SPOT-1644 (observed)')
ax.plot(aligned.index, aligned['Hs_mop'],  lw=0.7, alpha=0.9,
        color='darkorange', label='MOP B0385 (predicted)')
ax.set_ylabel('Hs (m)')
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)

ax = axes[1]
sc = ax.scatter(aligned['Hs_spot'], aligned['Hs_mop'],
                c=aligned['Dp_harvest'], cmap='hsv',
                alpha=0.15, s=4, vmin=150, vmax=340)
lim = max(aligned['Hs_spot'].max(), aligned['Hs_mop'].max()) * 1.05
ax.plot([0, lim], [0, lim], 'k--', lw=1, label='1:1')
ax.set_xlabel('SPOT-1644 Hs (m) -- observed')
ax.set_ylabel('MOP B0385 Hs (m) -- predicted')
ax.set_title(f'Hs scatter  |  R2={overall["R2"]:.3f}  bias={overall["pct_bias"]:.1f}%  (coloured by Harvest Dp)')
plt.colorbar(sc, ax=ax, label='Harvest Dp (deg)')

plt.tight_layout()
plt.show()

---
## 3. Train / Test Split

Training uses March-November 2025 (spring, summer, fall). The test set covers December 2025-April 2026 -- two months of pure winter the correction model has never seen. Holding out a winter period is deliberate: NW swell through the Channel Islands is the most important and most challenging condition for this site.

In [ ]:
cutoff = pd.Timestamp('2025-12-01', tz='UTC')

train = aligned[aligned.index <  cutoff].copy()
test  = aligned[aligned.index >= cutoff].copy()

print(f'Train: {len(train):,} hours  [{train.index[0].date()} -> {train.index[-1].date()}]')
print(f'Test:  {len(test):,}  hours  [{test.index[0].date()} -> {test.index[-1].date()}]')

baseline = skill_metrics(test['Hs_spot'].values, test['Hs_mop'].values,
                         'BASELINE -- uncorrected MOP on test set')

---
## 4. Correction Models

Three correction approaches are evaluated and compared against the baseline on the held-out winter test set:

1. **Direction-bin lookup table** -- per-bin scale factors from mean Hs ratios on the training set.
2. **Ridge regression** -- continuous predictor using sin/cos of Harvest direction and offshore Hs.
3. **Quantile mapping** -- maps MOP's CDF to SPOT's CDF per direction bin.

### Model 1: Direction-bin lookup table

In [ ]:
lookup = {}

print(f'{"Direction bin":<20} {"N_train":>8} {"scale_factor":>14}')
print('-' * 44)

for name, mask_fn in [
    ('NW  (290-330)', lambda df: (df['Dp_harvest'] >= 290) & (df['Dp_harvest'] < 330)),
    ('WNW (260-290)', lambda df: (df['Dp_harvest'] >= 260) & (df['Dp_harvest'] < 290)),
    ('W   (230-260)', lambda df: (df['Dp_harvest'] >= 230) & (df['Dp_harvest'] < 260)),
    ('SW  (200-230)', lambda df: (df['Dp_harvest'] >= 200) & (df['Dp_harvest'] < 230)),
    ('S   (150-200)', lambda df: (df['Dp_harvest'] >= 150) & (df['Dp_harvest'] < 200)),
]:
    sub   = train[mask_fn(train)]
    scale = sub['Hs_spot'].mean() / sub['Hs_mop'].mean() if len(sub) >= 10 else 1.0
    lookup[name] = scale
    print(f'{name:<20} {len(sub):>8,}  {scale:>13.4f}')

lookup['global'] = train['Hs_spot'].mean() / train['Hs_mop'].mean()
print(f'\nGlobal fallback: {lookup["global"]:.4f}')

In [ ]:
def apply_lookup(df, lookup):
    corrected = df['Hs_mop'].copy()
    dp = df['Dp_harvest']
    corrected[(dp >= 290) & (dp < 330)] *= lookup['NW  (290-330)']
    corrected[(dp >= 260) & (dp < 290)] *= lookup['WNW (260-290)']
    corrected[(dp >= 230) & (dp < 260)] *= lookup['W   (230-260)']
    corrected[(dp >= 200) & (dp < 230)] *= lookup['SW  (200-230)']
    corrected[(dp >= 150) & (dp < 200)] *= lookup['S   (150-200)']
    uncovered = ~((dp >= 150) & (dp < 330))
    corrected[uncovered] *= lookup['global']
    return corrected

test['Hs_lookup'] = apply_lookup(test, lookup)

m1 = skill_metrics(test['Hs_spot'].values, test['Hs_lookup'].values,
                   'Model 1: Direction-bin lookup table -- test set')

### Model 2: Ridge regression

In [ ]:
def make_features(df):
    dp_rad = np.radians(df['Dp_harvest'])
    return np.column_stack([
        np.sin(dp_rad),
        np.cos(dp_rad),
        df['Hs_harvest']
    ])

X_train = make_features(train)
X_test  = make_features(test)
ratio_train = train['Hs_spot'].values / train['Hs_mop'].values

scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_s, ratio_train)

test['Hs_ridge'] = test['Hs_mop'].values * ridge.predict(X_test_s)

m2 = skill_metrics(test['Hs_spot'].values, test['Hs_ridge'].values,
                   'Model 2: Ridge regression -- test set')

### Model 3: Quantile mapping

In [ ]:
def build_quantile_map(train, bin_mask):
    sub = train[bin_mask] if bin_mask.sum() >= 30 else train
    mop_sorted  = np.sort(sub['Hs_mop'].values)
    spot_sorted = np.sort(sub['Hs_spot'].values)
    return interp1d(mop_sorted, spot_sorted,
                    bounds_error=False,
                    fill_value=(spot_sorted[0], spot_sorted[-1]))

quantile_maps = {}
for name, mask_fn in [
    ('NW  (290-330)', lambda df: (df['Dp_harvest'] >= 290) & (df['Dp_harvest'] < 330)),
    ('WNW (260-290)', lambda df: (df['Dp_harvest'] >= 260) & (df['Dp_harvest'] < 290)),
    ('W   (230-260)', lambda df: (df['Dp_harvest'] >= 230) & (df['Dp_harvest'] < 260)),
    ('SW  (200-230)', lambda df: (df['Dp_harvest'] >= 200) & (df['Dp_harvest'] < 230)),
    ('S   (150-200)', lambda df: (df['Dp_harvest'] >= 150) & (df['Dp_harvest'] < 200)),
]:
    quantile_maps[name] = build_quantile_map(train, mask_fn(train))
quantile_maps['global'] = build_quantile_map(
    train, pd.Series([True]*len(train), index=train.index))

def apply_quantile(df, quantile_maps):
    corrected = df['Hs_mop'].copy().astype(float)
    dp = df['Dp_harvest']
    for name, (lo, hi) in [
        ('NW  (290-330)', (290, 330)),
        ('WNW (260-290)', (260, 290)),
        ('W   (230-260)', (230, 260)),
        ('SW  (200-230)', (200, 230)),
        ('S   (150-200)', (150, 200)),
    ]:
        mask = (dp >= lo) & (dp < hi)
        if mask.sum() > 0:
            corrected[mask] = quantile_maps[name](df.loc[mask, 'Hs_mop'].values)
    uncovered = ~((dp >= 150) & (dp < 330))
    if uncovered.sum() > 0:
        corrected[uncovered] = quantile_maps['global'](df.loc[uncovered, 'Hs_mop'].values)
    return corrected

test['Hs_quantile'] = apply_quantile(test, quantile_maps)

m3 = skill_metrics(test['Hs_spot'].values, test['Hs_quantile'].values,
                   'Model 3: Quantile mapping -- test set')

---
## 5. Results

The direction-bin lookup table outperforms both continuous correction approaches. The regression model fails to generalise due to seasonal mismatch between training (spring/summer) and test (winter). The quantile mapping performs similarly to the lookup table but adds complexity without gain.

The key finding is that MOP's errors are well-described as a direction-dependent mean offset. A simple scale factor per bin is sufficient.

In [ ]:
print('=' * 62)
print('RESULTS SUMMARY -- test set (Dec 2025 - Apr 2026)')
print('=' * 62)
print(f'{"Model":<37} {"R2":>6} {"Bias%":>8} {"RMSE%":>8}')
print('-' * 62)
for label, m in [
    ('Uncorrected MOP (baseline)',          baseline),
    ('Model 1: Direction-bin lookup table', m1),
    ('Model 2: Ridge regression',           m2),
    ('Model 3: Quantile mapping',           m3),
]:
    print(f'{label:<37} {m["R2"]:>6.3f} {m["pct_bias"]:>7.1f}% {m["pct_rmse"]:>7.1f}%')
print()
print('Winner: Model 1 -- direction-bin lookup table')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Correction model comparison -- test set (Dec 2025 - Apr 2026)',
             fontweight='bold')

obs_test = test['Hs_spot'].values
lim = 2.1

for ax, (pred, label, color) in zip(axes, [
    (test['Hs_mop'].values,     'Uncorrected MOP\nR2=0.415  bias=-19.9%', 'tomato'),
    (test['Hs_lookup'].values,  'Lookup table\nR2=0.582  bias=+1.5%',     'steelblue'),
    (test['Hs_ridge'].values,   'Ridge regression\nR2=0.375  bias=+10.8%','purple'),
]):
    ax.scatter(obs_test, pred, alpha=0.15, s=4, color=color)
    ax.plot([0, lim], [0, lim], 'k--', lw=1)
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.set_xlabel('SPOT observed Hs (m)')
    ax.set_ylabel('Predicted Hs (m)')
    ax.set_title(label)
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()

---
## 6. Confidence Flags

The corrected R2 per direction bin on the training set is used to assign a confidence flag reported alongside each forecast. NW swell is MEDIUM because the Channel Islands diffraction problem is most severe for this geometry. S and SW swells arrive through a more open swell window and receive HIGH confidence.

In [ ]:
confidence = {}

print(f'{"Direction bin":<20} {"N":>6} {"R2 corrected":>14} {"Confidence":>12}')
print('-' * 56)

for name, mask_fn in [
    ('NW  (290-330)', lambda df: (df['Dp_harvest'] >= 290) & (df['Dp_harvest'] < 330)),
    ('WNW (260-290)', lambda df: (df['Dp_harvest'] >= 260) & (df['Dp_harvest'] < 290)),
    ('W   (230-260)', lambda df: (df['Dp_harvest'] >= 230) & (df['Dp_harvest'] < 260)),
    ('SW  (200-230)', lambda df: (df['Dp_harvest'] >= 200) & (df['Dp_harvest'] < 230)),
    ('S   (150-200)', lambda df: (df['Dp_harvest'] >= 150) & (df['Dp_harvest'] < 200)),
]:
    sub = train[mask_fn(train)]
    if len(sub) < 10:
        continue
    o = sub['Hs_spot'].values
    p = sub['Hs_mop'].values * lookup[name]
    ss = 1 - np.sum((p - o)**2) / np.sum((o - np.mean(o))**2)
    flag = 'HIGH' if ss >= 0.35 else ('MEDIUM' if ss >= 0.15 else 'LOW')
    confidence[name] = {'R2': round(ss, 3), 'flag': flag}
    print(f'{name:<20} {len(sub):>6} {ss:>14.3f} {flag:>12}')

---
## 7. Export

The fitted lookup table and confidence flags are exported as `campus_point_correction.json`, loaded by `forecast.py` at runtime without retraining.

In [ ]:
correction_model = {
    'type':       'direction_bin_lookup',
    'trained_on': f'{train.index[0].date()} to {train.index[-1].date()}',
    'n_train':    len(train),
    'scale_factors': lookup,
    'confidence':    confidence,
    'bin_edges': {
        'NW  (290-330)': [290, 330],
        'WNW (260-290)': [260, 290],
        'W   (230-260)': [230, 260],
        'SW  (200-230)': [200, 230],
        'S   (150-200)': [150, 200],
    },
    'metrics': {
        'baseline_R2':    round(baseline['R2'], 3),
        'corrected_R2':   round(m1['R2'], 3),
        'baseline_bias':  round(baseline['pct_bias'], 1),
        'corrected_bias': round(m1['pct_bias'], 1),
    }
}

with open('campus_point_correction.json', 'w') as f:
    json.dump(correction_model, f, indent=2)

print('Saved: campus_point_correction.json')

---
## 8. WaveWatch III as Direction Predictor

Harvest provides offshore direction for the nowcast but cannot supply future directions for multi-day forecasting. WaveWatch III forecasts 7 days ahead. Here we test whether WW3 direction at a carefully chosen offshore point can substitute for Harvest direction without degrading correction skill.

The point at (34.5N, 238.5E = 121.5W) is the closest available non-masked WW3 grid cell to Harvest's actual position (34.458N, 120.782W).

In [ ]:
url = (
    "https://upwell.pfeg.noaa.gov/erddap/griddap/ww3_global.csv"
    "?Thgt[(2025-03-21T00:00:00Z):1:(2026-04-01T00:00:00Z)][(0.0):1:(0.0)][(34.5):1:(34.5)][(238.5):1:(238.5)]"
    ",Tdir[(2025-03-21T00:00:00Z):1:(2026-04-01T00:00:00Z)][(0.0):1:(0.0)][(34.5):1:(34.5)][(238.5):1:(238.5)]"
)

ww3 = pd.read_csv(url, skiprows=[1])
ww3['time'] = pd.to_datetime(ww3['time'], utc=True)
ww3 = ww3.set_index('time').sort_index()
ww3 = ww3[['Thgt', 'Tdir']].rename(columns={'Thgt': 'Hs', 'Tdir': 'Dp'}).dropna()

print(f'WW3 at 34.5N, 238.5E: {len(ww3):,} records')
print(f'Hs: {ww3["Hs"].min():.2f} - {ww3["Hs"].max():.2f}m')
print(f'Dp: {ww3["Dp"].min():.0f} - {ww3["Dp"].max():.0f} deg')

In [ ]:
def get_bin(dp):
    if 290 <= dp < 330: return 'NW'
    if 260 <= dp < 290: return 'WNW'
    if 230 <= dp < 260: return 'W'
    if 200 <= dp < 230: return 'SW'
    if 150 <= dp < 200: return 'S'
    return 'other'

aligned_ww3 = (
    aligned
    .join(ww3.rename(columns=lambda c: c + '_ww3'), how='inner')
    .dropna()
)

aligned_ww3['bin_harvest'] = aligned_ww3['Dp_harvest'].apply(get_bin)
aligned_ww3['bin_ww3']     = aligned_ww3['Dp_ww3'].apply(get_bin)

agreement = (aligned_ww3['bin_harvest'] == aligned_ww3['bin_ww3']).mean()
print(f'Aligned pairs: {len(aligned_ww3):,}')
print(f'Bin agreement (WW3 vs Harvest): {agreement:.1%}')
print()
print(pd.crosstab(aligned_ww3['bin_harvest'], aligned_ww3['bin_ww3']))

In [ ]:
test_ww3 = aligned_ww3[aligned_ww3.index >= cutoff].copy()

def apply_lookup_ww3(df, lookup):
    corrected = df['Hs_mop'].copy()
    dp = df['Dp_ww3']
    corrected[(dp >= 290) & (dp < 330)] *= lookup['NW  (290-330)']
    corrected[(dp >= 260) & (dp < 290)] *= lookup['WNW (260-290)']
    corrected[(dp >= 230) & (dp < 260)] *= lookup['W   (230-260)']
    corrected[(dp >= 200) & (dp < 230)] *= lookup['SW  (200-230)']
    corrected[(dp >= 150) & (dp < 200)] *= lookup['S   (150-200)']
    uncovered = ~((dp >= 150) & (dp < 330))
    corrected[uncovered] *= lookup['global']
    return corrected

test_ww3['Hs_corrected_ww3'] = apply_lookup_ww3(test_ww3, lookup)

m_ww3 = skill_metrics(test_ww3['Hs_spot'].values, test_ww3['Hs_corrected_ww3'].values,
                      'Correction using WW3 direction -- test set')
print()
print(f'Harvest-based correction: R2=0.582  bias=+1.5%')
print(f'WW3-based correction:     R2={m_ww3["R2"]:.3f}  bias={m_ww3["pct_bias"]:+.1f}%')
print()
print('Conclusion: WW3 direction is a viable drop-in for Harvest in the 7-day forecast pipeline.')

---
## 9. Conclusion

CDIP's MOP wave model consistently underpredicts wave height at Campus Point by about 20%. This is caused by the Channel Islands blocking and diffracting incoming swell -- physics that the linear refraction model cannot fully capture. The underprediction is direction-dependent: northwest swell arriving through Channel Islands gaps is most severely attenuated.

A direction-stratified lookup table that multiplies MOP output by a per-bin scale factor reduces bias from -20% to essentially zero and improves R2 from 0.41 to 0.58 on a held-out winter test set. More complex correction approaches (Ridge regression, quantile mapping) fail to outperform this simple model, confirming that MOP's error structure is a direction-dependent mean offset rather than a higher-order distributional problem.

WaveWatch III direction at the point nearest Harvest (34.5N, 238.5E) matches Harvest's bin classification 70.5% of the time and achieves equivalent correction skill (R2=0.586). This enables a 7-day forecast pipeline using WW3 forecast directions to drive the correction for future hours.

**Limitations:**
- Scale factors are derived from one spring/summer/fall cycle. A second full year will improve corrections for the underrepresented W and WNW bins.
- The WNW bin has LOW confidence (corrected R2=-0.15) due to seasonal mismatch between training and test conditions.
- The correction is calibrated for the SPOT-1644 buoy at B0385. The B0380 camera location (Goleta Point) is not directly covered.

| Model | R2 | Bias | RMSE |
|---|---|---|---|
| Uncorrected MOP | 0.415 | -19.9% | 26.3% |
| **Lookup table (Harvest direction)** | **0.582** | **+1.5%** | **27.8%** |
| Lookup table (WW3 direction) | 0.586 | +1.2% | 27.7% |
| Quantile mapping | 0.544 | +2.1% | 29.0% |
| Ridge regression | 0.375 | +10.8% | 32.4% |